# 07 Stage 2 Outcome Model — Data Preparation & Baseline Setup

**Purpose:** Integrates Stage 1 OOF barrier probabilities with health outcome targets (`target_unmet_fp`, `target_anc_gap`) and inspects Analytic Sub-samples for Stage 2 Modelling.
**Guide Reference:** Stage 2 Implementation Guide v2
**Depends on:** `07_data_integration.ipynb` (or `src/preprocessing/stage2_integration.py`)
**Outputs:** Summary statistics and baseline feature set validation for outcome modelling.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Standardized project root discovery
root = Path.cwd().resolve()
while root != root.parent and not (root / 'README.md').exists():
    root = root.parent
PROJECT_ROOT = root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.stage2_integration import build_stage2_dataset
print('Project Root:', PROJECT_ROOT)

In [ ]:
# Ensure Stage 2 integrated data is built
stage2_dir = PROJECT_ROOT / 'data' / 'processed' / 'stage2'
X_path = stage2_dir / 'X_stage2_preclustering.csv'
y_path = stage2_dir / 'y_stage2_targets.csv'
oof_path = stage2_dir / 'oof_barrier_probabilities.csv'

if not (X_path.exists() and y_path.exists() and oof_path.exists()):
    print('Building Stage 2 datasets via stage2_integration module...')
    build_stage2_dataset(project_root=PROJECT_ROOT)

X_stage2 = pd.read_csv(X_path)
y_stage2 = pd.read_csv(y_path)
oof_probs = pd.read_csv(oof_path)

print(f'X_stage2 preclustering matrix shape: {X_stage2.shape}')
print(f'y_stage2 targets matrix shape: {y_stage2.shape}')
print(f'OOF barrier probabilities shape: {oof_probs.shape}')

In [ ]:
# Analyze Stage 2 Target Analytic Samples
print('=== Stage 2 Target Distributions ===')
for target in y_stage2.columns:
    s = y_stage2[target]
    non_null_n = s.notna().sum()
    pos_n = s.dropna().sum()
    pos_rate = (pos_n / non_null_n) * 100 if non_null_n > 0 else 0
    print(f'Target: {target:20s} | Restricted N: {non_null_n:7,d} | Positives: {int(pos_n):6,d} ({pos_rate:.2f}%)')

In [ ]:
# Inspect OOF Barrier Probability Distributions
print('=== Out-of-Fold Barrier Probabilities Summary ===')
print(oof_probs.describe().T[['mean', 'std', 'min', '50%', 'max']])

In [ ]:
# Save Stage 2 Data Summary Artifact
output_dir = PROJECT_ROOT / 'outputs' / 'stage2_results'
output_dir.mkdir(parents=True, exist_ok=True)

summary_data = []
for target in y_stage2.columns:
    s = y_stage2[target]
    nn = s.notna().sum()
    pos = s.dropna().sum()
    summary_data.append({
        'Target': target,
        'Restricted_N': int(nn),
        'Positive_Count': int(pos),
        'Positive_Rate': float(pos / nn) if nn > 0 else 0.0
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(output_dir / 'stage2_data_summary.csv', index=False)
print('Saved Stage 2 Data Summary to:', output_dir / 'stage2_data_summary.csv')
summary_df